# Download the external training data

Only needed to retrain; building the submission from the committed model uses none of it.

* DFL / IDSSE (Bassek et al. 2025, CC BY 4.0, https://doi.org/10.6084/m9.figshare.28196177): the match information and
  raw position files of the seven matches (about 2.6 GB), from the figshare API, checked against its MD5 sums, to
  `data/external/dfl/`.
* Metrica (https://github.com/metrica-sports/sample-data): sample games 1-3 (about 180 MB) at a fixed commit, checked
  against the git blob hashes, to `data/external/metrica/Sample_Game_N/`.

`MTAH_EXTERNAL_DIR` replaces `data/external/`, as in the other notebooks. The notebook skips any file already present
with the right checksum, so you can rerun it after an interruption.

In [ ]:
import os
from pathlib import Path

PIPELINE_DIR = Path.cwd().resolve()     # notebooks run from possession/
assert (PIPELINE_DIR / "run_decoder.ipynb").is_file(), "run this notebook from the possession/ folder"
REPO_ROOT = PIPELINE_DIR.parent
import hashlib
import json
import urllib.request

EXTERNAL_DIR = Path(os.environ.get("MTAH_EXTERNAL_DIR") or (REPO_ROOT / "data" / "external"))
CHUNK = 1 << 20

FIGSHARE_ARTICLE = "https://api.figshare.com/v2/articles/28196177"
DFL_KINDS = ("matchinformation", "positions_raw_observed")      # the event files are not used

METRICA_COMMIT = "e706dd506b360d69d9d123d5b8026e7294b13996"
METRICA_RAW = "https://raw.githubusercontent.com/metrica-sports/sample-data/{}/data/{{}}".format(METRICA_COMMIT)
METRICA_FILES = {   # path under data/ -> git blob sha1
    "Sample_Game_1/Sample_Game_1_RawEventsData.csv": "3c426dd4f442dd433bccf89e3634f91aa310cd8f",
    "Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv": "ff409a5680fd75acc7cebcf786827214c687478b",
    "Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv": "ce2e91c0e633426cec1146f4e728b72417454e3d",
    "Sample_Game_2/Sample_Game_2_RawEventsData.csv": "decccd47c1d53960d290cb6679337eec83a8a97f",
    "Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv": "e7334d8d762954a215c49e55f211855000243fdc",
    "Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv": "8b4d3648888cc8b033e7142919ebaea04f6328e0",
    "Sample_Game_3/Sample_Game_3_events.json": "992b4cb502f0b0bc1244174f014f3f7ecb6d38ef",
    "Sample_Game_3/Sample_Game_3_metadata.xml": "74b55a80f6aeeb8f8df7fd26e4b3a94e2cc1738d",
    "Sample_Game_3/Sample_Game_3_tracking.txt": "7062b7ceaa44d53345a4625b8f447efc74895b1d",
}


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK), b""):
            digest.update(chunk)
    return digest.hexdigest()


def git_blob_sha(path):
    digest = hashlib.sha1(b"blob %d\0" % os.path.getsize(path))
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(url, target, checksum, expected):
    """Download url to target unless it is already there with the expected checksum."""
    name = os.path.relpath(target, EXTERNAL_DIR)
    if os.path.exists(target) and checksum(target) == expected:
        print("ok (present)  " + name, flush=True)
        return
    os.makedirs(os.path.dirname(target), exist_ok=True)
    partial = str(target) + ".part"
    with urllib.request.urlopen(url, timeout=120) as response, open(partial, "wb") as out:
        while True:
            chunk = response.read(CHUNK)
            if not chunk:
                break
            out.write(chunk)
    if checksum(partial) != expected:
        os.remove(partial)
        raise RuntimeError("checksum mismatch: " + name)
    os.replace(partial, target)
    print("ok            " + name, flush=True)

## DFL / IDSSE

In [ ]:
with urllib.request.urlopen(FIGSHARE_ARTICLE, timeout=60) as response:
    article = json.load(response)
files = [f for f in article["files"] if any(k in f["name"] for k in DFL_KINDS)]
print("DFL / IDSSE: {} files, {:.2f} GB, licence {}".format(
    len(files), sum(f["size"] for f in files) / 1e9, article["license"]["name"]), flush=True)
for f in files:
    fetch(f["download_url"], EXTERNAL_DIR / "dfl" / os.path.basename(f["name"]), md5, f["computed_md5"])

## Metrica sample games

In [ ]:
print("Metrica sample games 1-3, commit " + METRICA_COMMIT[:7], flush=True)
for rel, sha in METRICA_FILES.items():
    fetch(METRICA_RAW.format(rel), EXTERNAL_DIR / "metrica" / Path(rel), git_blob_sha, sha)
print("done:", EXTERNAL_DIR)